# 电商用户行为分析与购买预测

## 项目简介
基于真实电商平台（美妆品类）500万条用户行为日志，完成从数据清洗、SQL商业指标分析、RFM用户分层，
到机器学习购买预测的完整分析链路。

## 数据来源
Kaggle 公开数据集：[eCommerce Events History in Cosmetics Shop](https://www.kaggle.com/datasets/mkechinov/ecommerce-events-history-in-cosmetics-shop)

数据字段说明：
| 字段 | 含义 |
|------|------|
| event_time | 事件发生时间 |
| event_type | 事件类型（view/cart/remove_from_cart/purchase）|
| product_id | 商品ID |
| category_id | 类目ID |
| brand | 品牌 |
| price | 价格 |
| user_id | 用户ID |
| user_session | 会话ID |

## 分析目录
1. 数据清洗与入库
2. SQL商业指标分析（漏斗 / 时段 / RFM / 品牌）
3. 购买行为预测建模（特征工程 + 三模型对比）


## Part 1：数据清洗与入库

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'SimHei'
matplotlib.rcParams['axes.unicode_minus'] = False

# 加载数据（使用2019年10月数据，约400万条）
# 数据文件路径请根据本地实际路径修改
df = pd.read_csv('2019-Oct.csv')

print(f'原始数据量：{df.shape[0]} 行，{df.shape[1]} 列')
print()
print(df.info())


In [ ]:
# 查看缺失情况
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
pd.DataFrame({'缺失数量': missing, '缺失占比%': missing_pct})


In [ ]:
# 查看事件类型分布
print(df['event_type'].value_counts())
print()
# 查看价格分布
print(df['price'].describe())


In [ ]:
# ── 数据清洗 ──
# 1. category_code缺失98%，无分析价值，直接删除
df = df.drop(columns=['category_code'])

# 2. brand缺失40%，填充为'unknown'（缺失本身即一种状态，不捏造信息）
df['brand'] = df['brand'].fillna('unknown')

# 3. user_session缺失637条（0.02%），直接删行
df = df.dropna(subset=['user_session'])

# 4. 过滤price<=0的异常记录（现实中商品不可能为负价格）
df = df[df['price'] > 0]

# 5. 时间字段处理：转为datetime，提取时间特征
df['event_time'] = pd.to_datetime(df['event_time'])
df['date'] = df['event_time'].dt.date
df['hour'] = df['event_time'].dt.hour
df['day_of_week'] = df['event_time'].dt.dayofweek  # 0=周一，6=周日

print(f'清洗后数据量：{df.shape[0]} 行')
print(df.head())


In [ ]:
# 存入SQLite数据库，方便后续SQL分析
conn = sqlite3.connect('ecommerce.db')
df.to_sql('events', conn, if_exists='replace', index=False)
print('数据已存入SQLite，表名：events')


## Part 2：SQL 商业指标分析

### 2.1 购买漏斗分析

In [ ]:
# 漏斗分析：统计各步骤的独立用户数
sql_funnel = """
SELECT
    event_type,
    COUNT(DISTINCT user_id) AS user_count
FROM events
WHERE event_type IN ('view', 'cart', 'purchase')
GROUP BY event_type
"""

funnel = pd.read_sql(sql_funnel, conn)

order = {'view': 1, 'cart': 2, 'purchase': 3}
funnel['step'] = funnel['event_type'].map(order)
funnel = funnel.sort_values('step').reset_index(drop=True)

funnel['conversion_rate'] = (funnel['user_count'] / funnel['user_count'].shift(1) * 100).round(2)
funnel.loc[0, 'conversion_rate'] = 100.0
funnel['overall_rate'] = (funnel['user_count'] / funnel['user_count'].iloc[0] * 100).round(2)

print(funnel[['event_type', 'user_count', 'conversion_rate', 'overall_rate']])


In [ ]:
# 漏斗可视化
fig, ax = plt.subplots(figsize=(9, 5))

labels = [f'浏览\n{funnel.iloc[0].user_count:,}人',
          f'加购\n{funnel.iloc[1].user_count:,}人',
          f'购买\n{funnel.iloc[2].user_count:,}人']
colors = ['#4C72B0', '#DD8452', '#55A868']
bars = ax.barh(labels, funnel['user_count'], color=colors, height=0.5)

for i, (bar, row) in enumerate(zip(bars, funnel.itertuples())):
    label = '基准' if i == 0 else f'上一步转化率: {row.conversion_rate}%  |  整体转化率: {row.overall_rate}%'
    ax.text(bar.get_width() + 3000, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=10)

ax.set_xlabel('用户数')
ax.set_title('电商用户购买漏斗分析（2019年10月）')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# 关键结论
print(f'\n关键结论：')
print(f'  浏览→购买整体转化率：{funnel.iloc[2].overall_rate}%')
print(f'  加购→购买转化率：{funnel.iloc[2].conversion_rate}%')
print(f'  加购→购买转化率偏低，是主要流失节点，建议重点优化')


### 2.2 用户活跃时段分析

In [ ]:
sql_hour = """
SELECT
    hour,
    event_type,
    COUNT(*) AS event_count
FROM events
WHERE event_type IN ('view', 'cart', 'purchase')
GROUP BY hour, event_type
ORDER BY hour, event_type
"""

hour_df = pd.read_sql(sql_hour, conn)
hour_pivot = hour_df.pivot(index='hour', columns='event_type', values='event_count')

fig, ax = plt.subplots(figsize=(12, 5))
hour_pivot.plot(ax=ax, marker='o', linewidth=2)
ax.set_xlabel('小时（0~23点）')
ax.set_ylabel('事件次数')
ax.set_title('用户行为24小时分布（2019年10月）')
ax.set_xticks(range(0, 24))
ax.legend(title='行为类型')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

top_hour = hour_pivot['purchase'].idxmax()
print(f'购买行为最高峰：{top_hour}点')
print('用户从5点开始活跃，10-12点为浏览和购买双高峰，19点出现晚高峰')


### 2.3 RFM 用户价值分层

In [ ]:
# Step 1：用SQL提取每个用户的R/F/M原始值
sql_rfm = """
SELECT
    user_id,
    CAST(julianday('2019-10-31') - julianday(MAX(date)) AS INTEGER) AS recency,
    COUNT(*) AS frequency,
    ROUND(SUM(price), 2) AS monetary
FROM events
WHERE event_type = 'purchase'
GROUP BY user_id
"""

rfm = pd.read_sql(sql_rfm, conn)
print(f'有购买记录的用户数：{len(rfm)}')
print(rfm.describe().round(2))


In [ ]:
# Step 2：打分 + 分层
# qcut按分位数分组，保证每档人数大致相等
rfm['R_score'] = pd.qcut(rfm['recency'], q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

def classify_user(score):
    if score >= 13: return '高价值用户'
    elif score >= 10: return '中价值用户'
    elif score >= 7: return '低价值用户'
    else: return '流失风险用户'

rfm['segment'] = rfm['RFM_score'].apply(classify_user)

segment_summary = rfm.groupby('segment').agg(
    用户数=('user_id','count'),
    平均R=('recency','mean'),
    平均F=('frequency','mean'),
    平均M=('monetary','mean')
).round(2)
segment_summary['占比%'] = (segment_summary['用户数'] / len(rfm) * 100).round(2)
print(segment_summary)


In [ ]:
# RFM可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']

segment_counts = rfm['segment'].value_counts()
axes[0].pie(segment_counts, labels=segment_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[0].set_title('RFM 用户分层分布')

seg_order = ['高价值用户','中价值用户','低价值用户','流失风险用户']
valid_segs = [s for s in seg_order if s in segment_summary.index]
avg_m = [segment_summary.loc[s,'平均M'] for s in valid_segs]
axes[1].bar(valid_segs, avg_m, color=colors[:len(valid_segs)])
axes[1].set_title('各分层平均消费金额')
axes[1].set_ylabel('平均消费（美元）')
for i, v in enumerate(avg_m):
    axes[1].text(i, v+1, f'${v}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print('\n业务建议：')
print('  高价值用户（15%）：人均消费$109，重点维护，提供专属权益')
print('  流失风险用户（22%）：平均21天未访问，建议发送召回优惠')


### 2.4 品牌销售排行 TOP15

In [ ]:
sql_brand = """
SELECT
    brand,
    COUNT(DISTINCT user_id) AS buyer_count,
    COUNT(*) AS order_count,
    ROUND(SUM(price), 2) AS total_revenue,
    ROUND(AVG(price), 2) AS avg_price
FROM events
WHERE event_type = 'purchase' AND brand != 'unknown'
GROUP BY brand
ORDER BY total_revenue DESC
LIMIT 15
"""

brand_df = pd.read_sql(sql_brand, conn)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(brand_df['brand'][::-1], brand_df['total_revenue'][::-1], color='#3498db')
axes[0].set_title('品牌总销售额 TOP15')
axes[0].set_xlabel('销售额（美元）')

brand_sorted = brand_df.sort_values('avg_price', ascending=False)
axes[1].barh(brand_sorted['brand'][::-1], brand_sorted['avg_price'][::-1], color='#e74c3c')
axes[1].set_title('品牌平均客单价 TOP15')
axes[1].set_xlabel('平均价格（美元）')

plt.tight_layout()
plt.show()

print('洞察：strong品牌仅144名买家，但客单价$184.79，属于小众高端品牌，总销售额仍进入前10')


## Part 3：购买行为预测建模

**目标：** 基于用户当月行为特征，预测该用户是否会发生购买行为  
**方法：** 特征工程 + 逻辑回归 / 随机森林 / XGBoost 三模型对比  
**注意：** 本模型特征（行为数据）与标签（是否购买）均来自同一月份，
行为信号较强（加购行为与购买高度相关），因此AUC偏高属正常现象。
实际业务部署时应采用前N天行为预测后M天购买的时间窗口切分。


### 3.1 特征工程

In [ ]:
# 用pivot_table一次性统计各类行为次数（比逐行lambda快10倍）
event_counts = df.pivot_table(
    index='user_id', columns='event_type',
    values='product_id', aggfunc='count', fill_value=0
).reset_index()
event_counts.columns.name = None
event_counts = event_counts.rename(columns={
    'view':'view_count','cart':'cart_count',
    'purchase':'purchase_count','remove_from_cart':'remove_count'
})

# 其他维度特征
other = df.groupby('user_id').agg(
    unique_products=('product_id','nunique'),
    unique_brands=('brand','nunique'),
    active_days=('date','nunique'),
    avg_price=('price','mean'),
    peak_hour=('hour', lambda x: x.mode()[0])
).reset_index()

# 合并
feature_df = event_counts.merge(other, on='user_id')

# 目标变量（注意：不把purchase_count和total_spent放入特征，避免数据泄露）
feature_df['label'] = (feature_df['purchase_count'] > 0).astype(int)

print(f'特征表大小：{feature_df.shape}')
print(f'购买用户占比：{feature_df["label"].mean()*100:.2f}%  （存在类别不平衡，需特殊处理）')
print(feature_df.head())


### 3.2 模型训练

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
import numpy as np

# 特征列（已排除purchase_count和total_spent，防止数据泄露）
features = ['view_count','cart_count','remove_count',
            'unique_products','unique_brands','active_days',
            'avg_price','peak_hour']

X = feature_df[features]
y = feature_df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'训练集：{X_train.shape[0]}条  测试集：{X_test.shape[0]}条')
print(f'正负样本比 ≈ 1:{round((y_train==0).sum()/(y_train==1).sum())}  → 使用加权策略处理不平衡')

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# 逻辑回归：class_weight='balanced' 自动加权少数类
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)

# 随机森林
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# XGBoost：scale_pos_weight = 负样本数/正样本数
neg_pos = (y_train==0).sum() / (y_train==1).sum()
xgb = XGBClassifier(scale_pos_weight=neg_pos, n_estimators=100,
                    random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)

print('三个模型训练完成')


### 3.3 模型评估

In [ ]:
models = {'逻辑回归':(lr,X_test_sc), '随机森林':(rf,X_test), 'XGBoost':(xgb,X_test)}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
auc_scores = {}

for name, (model, X) in models.items():
    y_prob = model.predict_proba(X)[:,1]
    auc = roc_auc_score(y_test, y_prob)
    auc_scores[name] = auc
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})')

axes[0].plot([0,1],[0,1],'k--',label='随机猜测 (AUC=0.5)')
axes[0].set_xlabel('假正率 (FPR)')
axes[0].set_ylabel('真正率 (TPR)')
axes[0].set_title('三模型 ROC 曲线对比')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

names, aucs = list(auc_scores.keys()), list(auc_scores.values())
bars = axes[1].bar(names, aucs, color=['#3498db','#2ecc71','#e74c3c'], width=0.4)
axes[1].set_ylim(0.5, 1.0)
axes[1].set_title('三模型 AUC 分数对比')
axes[1].set_ylabel('AUC Score')
for bar, val in zip(bars, aucs):
    axes[1].text(bar.get_x()+bar.get_width()/2, val+0.005,
                 f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# 最优模型详细报告
best_name = max(auc_scores, key=auc_scores.get)
best_model, best_X = models[best_name]
y_pred = best_model.predict(best_X)
print(f'\n最优模型：{best_name}（AUC={auc_scores[best_name]:.4f}）')
print('\n分类报告：')
print(classification_report(y_test, y_pred, target_names=['未购买','购买']))
print('说明：Precision偏低（37%）是严重类别不平衡下的正常现象，Recall=97%说明模型几乎不漏掉真实买家')


### 3.4 特征重要性分析

In [ ]:
feature_names_cn = ['浏览次数','加购次数','移除购物车次数',
                    '浏览商品种数','接触品牌数','活跃天数',
                    '平均价格','最常活跃时段']

importances = xgb.feature_importances_
sorted_idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.barh([feature_names_cn[i] for i in sorted_idx],
               importances[sorted_idx], color='#3498db')
for bar, val in zip(bars, importances[sorted_idx]):
    ax.text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
ax.set_xlabel('特征重要性得分')
ax.set_title('XGBoost 特征重要性排名\n（预测用户是否购买）')
plt.tight_layout()
plt.show()

print('\n业务建议：')
print('  加购次数（0.787）是最强预测信号')
print('  基于模型输出的购买概率，可将用户分为高/中/低意向层：')
print('  对中意向层（概率30%-70%）定向推送优惠券，避免对高意向用户浪费营销资源')
